# 07. Hands-on: Strands Decider — the Strands team's dedicated decision model

[strands-decider](https://github.com/strands-labs/strands-decider) is a **dedicated decision model** built by the Strands Agents team (Apache-2.0, HF `StrandsAgents` org — mirrored in the `amazon` org).

- Architecture: take a pretrained LLM torso (Qwen3.5-2B-Base or Gemma 4), **remove the LM head, add a ~1M-parameter pointer head**, adapt with LoRA — it answers in a single forward pass with no generation.
- Question types: noul (yes/no probability), choice (N-way), score (ordinal scale) + **calibrated confidence** (per the README: on unseen short classification tasks, answers at confidence >= 0.9 are right about 95% of the time).
- Server mode speaks **`POST /v1/systemone`** (the Jev/System One wire format) — the **same format** as the SGLang endpoint we have been using, so it plugs into our workflow with the same client.
- Official Strands Agents SDK integration: an `InterventionHandler` example asks the decider "are the arguments grounded / is this call premature?" right before each tool call and returns Proceed/Guide.

### Starting the server (in this workshop environment a dedicated CPU EC2 (c7i.2xlarge) already serves it on :8002)
```bash
pip install strands-decider            # a dedicated venv is recommended
strands-decider serve \
  StrandsAgents/strands-decider-2B-qwen3.5-v1-2610 --port 8002 --device cuda
```


In [ ]:
import os
os.environ["DMWS_LANG"] = "en"   # localize helper output/graph captions (set BEFORE importing ws_common)
import sys; from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # ws_common path (relative inside the repo)
import json, urllib.request
import ws_common as W

print(json.dumps(json.load(urllib.request.urlopen(f"http://{W.STRANDS_HOST}:8002/health", timeout=5)), indent=1))


## 1. Single decisions

Because the wire format is identical, you just point the **same `SGLangDecision` client at port 8002**. We ask D1/D2 with "My wallet was stolen. Please freeze my card."


In [ ]:
from agent.decisions import SGLangDecision
from agent.prompts import D1_QUESTION, D2_QUESTION
from agent.state import CATEGORIES, D1_CHOICES

strands_dec = SGLangDecision(model="strands-decider", port=8002)   # reuse the same client
msg = "My wallet was stolen. Please freeze my card."
for q, ch, did in ((D1_QUESTION, D1_CHOICES, "d1"), (D2_QUESTION, CATEGORIES, "d2")):
    r = strands_dec.decide(state_text=msg, question=q, choices=ch, decision_id=did)
    print(f"[{did}] -> {r.label:<16s} confidence {r.confidence:.3f}   {r.latency_ms:,.1f}ms")

**Observation**: D2 (routing) picks card_security with solid confidence (~0.70), while **D1 comes out noticeably less certain (~0.60)** on the 3-way safety call. That is the value of calibrated confidence — the model tells you in numbers how sure it is, so you can design a **cascade that escalates to a larger model below a confidence threshold**. (The Qwen3-VL you saw in 04 saturates at conf 1.0 even when it over-blocks, which makes that design impossible.)


## 2. Plugging it straight into our LangGraph workflow

Any decision backend with a matching `.decide()` signature is swappable. **Changing only the client's port — no code changes** — we run scenario 2 (delayed overseas transfer, the tool retry loop) on strands-decider decisions.

> Why scenario 2 instead of scenario 1 ("my card is not swiping"): strands-decider's D1 **mis-blocks** the scenario-1 question as out_of_scope (conf 0.498), ending the workflow after 2 STEPs. Try swapping it yourself — it is the lived experience of the D1 0.613 number in the quality table below.


In [ ]:
from agent.graph_v2 import build_graph_v2

client, llm = W.make_clients()
graph_strands = build_graph_v2(strands_dec, llm)                     # decisions: strands-decider
graph_sglang  = build_graph_v2(SGLangDecision(port=8001), llm)       # decisions: SGLang Qwen3-VL (baseline)
sc = W.scenario(2)
print("Scenario:", sc["title"], "|", sc["question"])

In [ ]:
res_strands = W.run_verbose(graph_strands, sc["question"], sc["script"], "StrandsDecider")

In [ ]:
res_sglang = W.run_verbose(graph_sglang, sc["question"], sc["script"], "SGLang")

In [ ]:
W.compare(res_strands, res_sglang)

## Measured decision quality on the Korean bench (3-backend comparison)

D1/D2 measured over the workshop's 160-conversation Korean bench (the [KR] track inputs) with objective gold labels (`scripts/95_decider_quality_bench.py`, `results/summary/decider_quality_ko.json`):

| backend | D1 guardrail (3-way) | D2 routing strict/lenient (12-way) | p50 latency |
|---|---|---|---|
| SGLang Qwen3-VL-8B | **0.913** | **0.629 / 0.707** | 33ms |
| strands-decider 2B | 0.613 | 0.500 / 0.593 | 85ms |
| Laya multilingual 322M | 0.419 | 0.357 / 0.379 | 21ms |

**Reading**: model capacity dominated the quality ceiling (8B > 2B > 0.3B), beating language specialization (Laya multilingual). The Laya number matches its own documented zero-shot score (0.362); Laya is a **"fine-tune me" model** — RLCD fine-tuning lifts it to 0.766. Conversely, smaller means better latency and footprint — a clean three-way quality/cost/speed trade-off.


## 3. Wrap-up — where is this the right choice?

| Angle | strands-decider (dedicated 2B) | SGLang + Qwen3-VL (endpoint added to a served LLM) |
|---|---|---|
| Deployment | One small dedicated model (CPU-capable, ~489ms/question per README) | Reuses the LLM/VLM you already serve (~zero marginal cost) |
| Confidence | **Calibrated** (low confidence = genuinely unsure) -> easy cascade design | Tends to saturate (hard to design thresholds) |
| Multimodal | Text only (vision=false) | Image decisions possible (lab 05) |
| Ecosystem | Official Strands Agents SDK intervention example | Broad System One wire-format compatibility |

- Decision latency in this environment (GPU shared with SGLang) is **~80ms** warm — slower than SGLang (~35ms) but far faster than Bedrock LLM decisions (~640ms).
- On the Korean bench the 2B torso is "usable but unsure of itself" — and the confidence says so honestly, which makes a **low threshold + larger-model fallback** the natural setup for Korean decisions.
